In [2]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

# agiso@dtu.dk
using JuMP, HiGHS


   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`


In [11]:
##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

nodes = 6

arches = zeros(Int, nodes, nodes)
cost = zeros(Int, nodes, nodes)


arches[1,2], cost[1,2] = 1, 6
arches[1,5], cost[1,5] = 1, 2

arches[2,3], cost[2,3] = 1, 5
arches[2,6], cost[2,6] = 1, 2

arches[3,4], cost[3,4] = 1, 5

arches[5,2], cost[5,2] = 1, 5
arches[5,3], cost[5,3] = 1, 2
arches[5,6], cost[5,6] = 1, 5

arches[6,3], cost[6,3] = 1, 6
arches[6,4], cost[6,4] = 1, 10

source = 1
sink = 4

########## ---------- Variables ---------- ##########
@variable(model, x[1:nodes, 1:nodes], Bin)

########## ---------- Objective ---------- ##########
@objective(model, Min, sum(x[i, j] * cost[i,j] for i in 1:nodes, j in 1:nodes) )

########## ---------- Constrait ---------- ##########
# Flow constraint for intermediate nodes
# from == to
@constraint(model, [k in 1:nodes; k != source && k != sink],
    sum(x[k,j] for j in 1:nodes) - sum(x[j,k] for j in 1:nodes) == 0
)

# Flow for source
# from - to = 1
@constraint(model,
    sum(x[source,j] for j in 1:nodes) - sum(x[j,source] for j in 1:nodes) == 1
)

# Flow for sink
# from - to = -1
@constraint(model,
    sum(x[sink,j] for j in 1:nodes) - sum(x[j,sink] for j in 1:nodes) == -1
)

# If it's not an edge, don't use it
@constraint(model, [i in 1:nodes, j in 1:nodes],
    x[i,j] <= arches[i,j]
)

########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))

for i in 1:nodes
    println(value.(x[i, :]))
end

Optimal solution:
9.0
[0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
arches = zeros(Int, nodes, nodes)
cost   = zeros(Int, nodes, nodes)
nodes = 10

# From node 4
arches[4,3],  cost[4,3]   = 1, 8
arches[4,5],  cost[4,5]   = 1, 5


# From node 3
arches[3,6],  cost[3,6]   = 1, 9
arches[3,10], cost[3,10]  = 1, 9
arches[3,1],  cost[3,1]   = 1, 3
arches[3,8],  cost[3,8]   = 1, 8

# From node 6
arches[6,9],  cost[6,9]   = 1, 3

# From node 5
arches[5,9],  cost[5,9]   = 1, 7

# From node 8
arches[8,10], cost[8,10]  = 1, 5
arches[8,4],  cost[8,4]   = 1, 8

# From node 9
arches[9,10], cost[9,10]  = 1, 6
arches[9,7],  cost[9,7]   = 1, 1
arches[9,1],  cost[9,1]   = 1, 7

# From node 10
arches[10,7], cost[10,7]  = 1, 7

# From node 7
arches[7,2],  cost[7,2]   = 1, 1
arches[7,1],  cost[7,1]   = 1, 8

# From node 1
arches[1,5],  cost[1,5]   = 1, 5
arches[1,4],  cost[1,4]   = 1, 4
